# Final Project Experiments

## Setup

In [ ]:
import cv2 as cv
import os
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import csv
from matplotlib import pyplot as plt

INPUT_FILEPATH = "data/input/"
OUTPUT_FILEPATH = "data/output/"
PRED_ZERODCE_FILEPATH = "data/predicted/zero_dcepp/"
PRED_ATTN_FILEPATH = "data/predicted/attention_zero_dcepp/"
HISTEQ_FILEPATH = "data/histeq/"

RAND_SEED = 6524
np.random.seed(RAND_SEED)

NOISE_GAMMA = 3.0                   # Fixed gamma for noise experiments
NOISE_SIGMAS = [0.0125, 0.025, 0.05]   # Sigma values for gaussian noise

DARKEN_GAMMAS = [2.0, 3.0, 4.0]     # Gamma values for synthetic darkening

## Helper Functions and Experiments

In [2]:
def add_noise(image, sigma):
    # Rescale image to [0, 1]
    rescaled_image = image / 255

    # Generate Gaussian noise
    noise = np.random.normal(0, sigma, (image.shape[0], image.shape[1], image.shape[2]))

    noisy_image = rescaled_image + noise                    # Add noise
    noisy_image = noisy_image * 255.0                       # Scale image back to [0, 255]
    noisy_image = np.clip(noisy_image, 0, 255)              # Clip to range [0, 255]
    noisy_image = np.round(noisy_image).astype(np.uint8)    # Convert to integers
    
    return noisy_image

def gamma_adjust(image, gamma):

    # Convert image to HSV
    hsv_image = cv.cvtColor(image, cv.COLOR_BGR2HSV)
    
    # Scale intensity values to [0, 1]
    scaled_intensity = hsv_image[:, :, 2] / 255

    gamma_intensity = np.power(scaled_intensity, gamma)           # Apply gamma correction

    gamma_intensity = gamma_intensity * 255.0                       # Scale image back to [0, 255]
    gamma_intensity = np.clip(gamma_intensity, 0, 255)              # Clip to range [0, 255]
    gamma_intensity = np.round(gamma_intensity).astype(np.uint8)    # Convert to integers

    hsv_image[:, :, 2] = gamma_intensity

    output = cv.cvtColor(hsv_image, cv.COLOR_HSV2BGR)

    return output

def apply_histeq(image):
    hsv_image = cv.cvtColor(image, cv.COLOR_BGR2HSV)

    new_V = cv.equalizeHist(hsv_image[:, :, 2])

    hsv_image[:, :, 2] = new_V

    output = cv.cvtColor(hsv_image, cv.COLOR_HSV2BGR)

    return output

def noise_experiments(image, name, ext):

    for sigma in NOISE_SIGMAS:
        
        gamma_image = gamma_adjust(image, NOISE_GAMMA)  # Apply gamma correction to darken image
        noisy_image = add_noise(gamma_image, sigma)     # Add noise
        histeq_image = apply_histeq(noisy_image)

        # Ouput with corresponding name
        output_name = "{}$noise_sigma_{}.{}".format(name, str(sigma).replace(".", "-"), ext)
        cv.imwrite(OUTPUT_FILEPATH + output_name, noisy_image)
        cv.imwrite(HISTEQ_FILEPATH + output_name, histeq_image)

# Gamma values to be tested
def gamma_experiments(image, name, ext):

    for gam in DARKEN_GAMMAS:
        gamma_image = gamma_adjust(image, gam)  # Apply gamma correction to darken image
        histeq_image = apply_histeq(gamma_image)

        # Ouput with corresponding name
        output_name = "{}$gamma_{}.{}".format(name, str(gam).replace(".", "-"), ext)
        cv.imwrite(OUTPUT_FILEPATH + output_name, gamma_image)
        cv.imwrite(HISTEQ_FILEPATH + output_name, histeq_image)


## Running Experiments

In [3]:
# Remove any existing files
existing_files = os.listdir(OUTPUT_FILEPATH)
existing_files.remove(".gitignore") # Don't remove .gitignore

for file in existing_files:
    os.remove(OUTPUT_FILEPATH + file)

histeq_files = os.listdir(HISTEQ_FILEPATH)
histeq_files.remove(".gitignore")

for file in histeq_files:
    os.remove(HISTEQ_FILEPATH + file)

# Processing images
input_filenames = os.listdir(INPUT_FILEPATH)
input_filenames.remove(".gitignore") # Don't try to process .gitignore as an image

for name_file in input_filenames:

    name, ext = name_file.split(".")

    image = cv.imread(INPUT_FILEPATH + name_file)

    noise_experiments(image, name, ext)
    gamma_experiments(image, name, ext)

## Evaluate Predicted Outputs
Run tests before proceeding

### Create dictionaries to hold data

In [4]:
noise_exp_keys = list()
gamma_exp_keys = list()

zerodce_dict = dict()
zerodce_metrics = dict()

attn_dict = dict()
attn_metrics = dict()

histeq_dict = dict()
histeq_metrics = dict()

for sigma in NOISE_SIGMAS:
    key = "noise_sigma_{}".format(str(sigma).replace(".", "-"))
    noise_exp_keys.append(key)

    zerodce_dict[key] = dict()
    zerodce_metrics[key] = dict()

    attn_dict[key] = dict()
    attn_metrics[key] = dict()

    histeq_dict[key] = dict()
    histeq_metrics[key] = dict()

for gam in DARKEN_GAMMAS:
    key = "gamma_{}".format(str(gam).replace(".", "-"))
    gamma_exp_keys.append(key)
    
    zerodce_dict[key] = dict()
    zerodce_metrics[key] = dict()

    attn_dict[key] = dict()
    attn_metrics[key] = dict()

    histeq_dict[key] = dict()
    histeq_metrics[key] = dict()


### Load Images into Memory

In [5]:
input_filenames = os.listdir(INPUT_FILEPATH)
input_filenames.remove(".gitignore")    # Skip .gitignore

# Load input images into memory
original_images = dict()

for name_file in input_filenames:
    name, ext = name_file.split(".")

    in_image = cv.imread(INPUT_FILEPATH + name_file)

    original_images[name] = in_image

# Load predicted zero-dce images into memory
pred_zero_filenames = os.listdir(PRED_ZERODCE_FILEPATH)
# pred_filenames.remove(".gitignore")     # Skip .gitignore

for name_file in pred_zero_filenames:
    name, ext = name_file.split(".")
    original_name, exp_key = name.split("$")

    pred_image = cv.imread(PRED_ZERODCE_FILEPATH + name_file)
    
    zerodce_dict[exp_key][original_name] = pred_image

# Load predicted zero-dce + attention images into memory
pred_attn_filenames = os.listdir(PRED_ATTN_FILEPATH)

for name_file in pred_attn_filenames:
    name, ext = name_file.split(".")
    original_name, exp_key = name.split("$")

    pred_image = cv.imread(PRED_ATTN_FILEPATH + name_file)
    
    attn_dict[exp_key][original_name] = pred_image


# Load histogram equalized images into memory
histeq_filenames = os.listdir(HISTEQ_FILEPATH)
histeq_filenames.remove(".gitignore")  # Skip .gitignore

for name_file in histeq_filenames:
    name, ext = name_file.split(".")
    original_name, exp_key = name.split("$")

    histeq_image = cv.imread(HISTEQ_FILEPATH + name_file)

    histeq_dict[exp_key][original_name] = histeq_image

### Calculate metrics

In [6]:
def calc_metrics(exp_dict, exp_metrics):
    # Traverse through each experiment
    for exp_key, pred_images in exp_dict.items():

        # Create a dictionary for each metric
        exp_metrics[exp_key]["PSNR"] = dict()
        exp_metrics[exp_key]["SSIM"] = dict()

        # Traverse through all of the images
        for image_name, original_image in original_images.items():
            
            # Retrieve predicted image
            pred_image = pred_images[image_name]

            # Save metrics
            exp_metrics[exp_key]["PSNR"][image_name] = peak_signal_noise_ratio(original_image, pred_image)
            exp_metrics[exp_key]["SSIM"][image_name] = structural_similarity(original_image, pred_image, channel_axis=2)

calc_metrics(zerodce_dict, zerodce_metrics)
calc_metrics(attn_dict, attn_metrics)
calc_metrics(histeq_dict, histeq_metrics)

### Output data CSV

In [ ]:
def write_metrics(metrics_dict, writer, row_name):
    for exp_key, metrics in metrics_dict.items():
        metric_list = ["{} {}".format(row_name, exp_key)]

        for metric_name, metric_values in metrics.items():
            
            values = list(metric_values.values())
            mean_value = np.mean(values).round(5)
            median_value = np.median(values).round(5)

            metric_list.append(mean_value)
            metric_list.append(median_value)

        writer.writerow(metric_list)

output_csv = open("output_metrics.csv", "w", newline="")

writer = csv.writer(output_csv)

header = ["", "Mean PSNR", "Median PSNR", "Mean SSIM", "Median SSIM"]
writer.writerow(header)

write_metrics(zerodce_metrics, writer, "(Zero-DCE)")
write_metrics(attn_metrics, writer, "(Zero-DCE + Attn.)")
write_metrics(histeq_metrics, writer, "(Hist. Eq.)")

output_csv.close()